In [1]:
import joblib
import json
import shap
import pandas as pd
import numpy as np
import lightgbm as lgb
import matplotlib.pyplot as plt
import japanize_matplotlib
from typing import Tuple, Any
from sklearn.model_selection import StratifiedKFold

/opt/anaconda3/envs/StudentCup26env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [3]:
from src.data import load_train, load_test, convert_category
from src.config import load_config
from src.cv_features import create_cv_features

In [10]:
def calculate_shap(config_name: str, exp_dir_name: str) -> Tuple[np.ndarray, pd.DataFrame, pd.Series, pd.Series]:
    
    config = load_config(f"../configs/{config_name}.yaml")
    exp_name = config["experiment"]["name"]
        
        # モデルと特徴量、メトリクス、OOF予測値の読み込み
    model = joblib.load(f"../outputs/{exp_dir_name}/model.pkl")
    features = joblib.load(f"../outputs/{exp_dir_name}/feature_columns.pkl")
        
    with open(f"../outputs/{exp_dir_name}/metrics.json", "r", encoding="utf-8") as f:
        metrics = json.load(f)
            
    oof = pd.read_csv(f"../outputs/{exp_dir_name}/oof.csv")
        
        # データのロードと前処理
    train = load_train()
    test = load_test()
        
    target = config["data"]["target"]
    id_col = config["data"]["id"]
    drop_cols = config["feature"]["drop_columns"] + [target] + [id_col]
    cat_features = config["feature"]["categorical_features"]
        
    train, test = convert_category(train, test, cat_features)
        
    X = train.drop(columns=drop_cols)
    y = train[target]
    X = X[features]

    lgb_model = model.models[0]
    explainer = shap.TreeExplainer(lgb_model)
    shap_values = explainer.shap_values(X)

    return shap_values, X, y, oof

In [13]:
def run_shap_analysis(config_name: str, exp_dir_name: str) -> None:
    # 設定ファイルの読み込み
    config = load_config(f"../configs/{config_name}.yaml")
    exp_name = config["experiment"]["name"]
    
    # モデルと特徴量、メトリクス、OOF予測値の読み込み
    model = joblib.load(f"../outputs/{exp_dir_name}/model.pkl")
    features = joblib.load(f"../outputs/{exp_dir_name}/feature_columns.pkl")
    
    with open(f"../outputs/{exp_dir_name}/metrics.json", "r", encoding="utf-8") as f:
        metrics = json.load(f)
        
    oof = pd.read_csv(f"../outputs/{exp_dir_name}/oof.csv")
    
    # データのロードと前処理
    train = load_train()
    test = load_test()
    
    target = config["data"]["target"]
    id_col = config["data"]["id"]
    drop_cols = config["feature"]["drop_columns"] + [target] + [id_col]
    cat_features = config["feature"]["categorical_features"]
    
    train, test = convert_category(train, test, cat_features)
    
    X = train.drop(columns=drop_cols)
    y = train[target]
    X = X[features]
    
    # SHAP値の計算
    lgb_model = model.models[0]
    explainer = shap.TreeExplainer(lgb_model)
    shap_values = explainer.shap_values(X)
    
    # 個別のドットプロット生成・保存
    shap.summary_plot(shap_values, X, show=False)
    plt.savefig(f'images/{exp_name}_shap.png')
    plt.close()
    
    shap.summary_plot(shap_values[y == 1], X[y == 1], show=False)
    plt.savefig(f'images/{exp_name}_購入企業_shap.png')
    plt.close()
    
    # 見逃し (False Negative) の抽出とプロット
    fn = (y == 1) & (oof["prediction"] < metrics["best_threshold"])
    
    shap.summary_plot(shap_values[fn], X[fn], show=False)
    plt.savefig(f'images/{exp_name}_FN_shap.png')
    plt.close()
    
    # 誤検出　　(False Positive)　　の抽出
    fp = (y==0) & (oof["prediction"] > metrics["best_threshold"])

    shap.summary_plot(shap_values[fp], X[fp], show=False)
    plt.savefig(f'images/{exp_name}_FP_shap.png')
    plt.close()


    # バープロット比較図の生成・保存　
    fig, axes = plt.subplots(1, 4, figsize=(20, 15))
    
    plt.sca(axes[0])
    shap.summary_plot(shap_values, X, plot_type="bar", show=False)
    axes[0].set_title("All Data")
    
    plt.sca(axes[1])
    shap.summary_plot(shap_values[y == 1], X[y == 1], plot_type="bar", show=False)
    axes[1].set_title("y == 1")
    
    plt.sca(axes[2])
    shap.summary_plot(shap_values[fn], X[fn], plot_type="bar", show=False)
    axes[2].set_title("False Negative")

    plt.sca(axes[3])
    shap.summary_plot(shap_values[fp], X[fp], plot_type="bar", show=False)
    axes[3].set_title("False_Positive")

    plt.subplots_adjust(left=0.25, wspace=0.4)
    plt.savefig(f'images/{exp_name}_特徴量重要度比較.png')
    plt.close()

In [4]:
def run_oof_shap_analysis(config_name: str, exp_dir_name: str) -> None:
    # 1. 設定ファイルと学習済み成果物のロード
    config = load_config(f"../configs/{config_name}.yaml")
    exp_name = config["experiment"]["name"]

    model = joblib.load(f"../outputs/{exp_dir_name}/model.pkl")
    features = joblib.load(
        f"../outputs/{exp_dir_name}/feature_columns.pkl"
    )  # 学習時の79列

    with open(
        f"../outputs/{exp_dir_name}/metrics.json", "r", encoding="utf-8"
    ) as f:
        metrics = json.load(f)

    oof = pd.read_csv(f"../outputs/{exp_dir_name}/oof.csv")

    # 2. データのロードと前処理の設定
    train = load_train()
    target = config["data"]["target"]
    id_col = config["data"]["id"]
    cat_features = config["feature"]["categorical_features"]
    
    # 学習時と同じ不要カラムのリストを作成
    drop_cols = config["feature"]["drop_columns"] + [target] + [id_col]

    # 1) カテゴリ変換を行う
    train, _ = convert_category(train, train, cat_features)

    # 2) 学習時（train_cv.py）と同様に drop_cols をすべて削除して X_raw を作成
    X_raw = train.drop(columns=drop_cols)
    y_raw = train[target]

    n_splits = config.get("train", {}).get("n_splits", 5)
    random_state = config.get("train", {}).get("random_state", 42)
    skf = StratifiedKFold(
        n_splits=n_splits, shuffle=True, random_state=random_state
    )

    oof_shap_list = []
    oof_X_list = []
    oof_y_list = []
    oof_idx_list = []

    # 3. 各 Fold ごとに Validation データに対する SHAP を計算
    for fold, (train_idx, val_idx) in enumerate(skf.split(X_raw, y_raw)):
        X_tr_raw = X_raw.iloc[train_idx].copy()
        X_val_raw = X_raw.iloc[val_idx].copy()
        y_val = y_raw.iloc[val_idx].copy()

        # --- 特徴量生成 (学習時と同じ入力状態で実行) ---
        X_tr, stats_dict = create_cv_features(X_tr_raw)
        X_val, _ = create_cv_features(X_val_raw, stats_dict=stats_dict)

        # --- 最終調整: 保存しておいた学習時カラム (79列) だけを抽出し、順序を一致させる ---
        X_val = X_val[features]

        # 対応する Fold のモデルで SHAP 値を計算
        fold_model = model.models[fold]
        explainer = shap.TreeExplainer(fold_model)
        shap_vals = explainer.shap_values(X_val)

        if isinstance(shap_vals, list):
            shap_vals = shap_vals[1]

        oof_shap_list.append(shap_vals)
        oof_X_list.append(X_val)
        oof_y_list.append(y_val)
        oof_idx_list.append(val_idx)

    # 4. 全 Fold の結果を結合し、元データのインデックス順に整列
    all_indices = np.concatenate(oof_idx_list)
    reorder_idx = np.argsort(all_indices)

    shap_values = np.vstack(oof_shap_list)[reorder_idx]
    X = pd.concat(oof_X_list, axis=0).iloc[reorder_idx].reset_index(drop=True)
    y = pd.concat(oof_y_list, axis=0).iloc[reorder_idx].reset_index(drop=True)

    # 5. 各種 SHAP プロットの生成と保存
    # 個別のドットプロット生成・保存
    shap.summary_plot(shap_values, X, show=False)
    plt.savefig(f"images/{exp_name}_shap.png")
    plt.close()

    shap.summary_plot(shap_values[y == 1], X[y == 1], show=False)
    plt.savefig(f"images/{exp_name}_購入企業_shap.png")
    plt.close()

    # 見逃し (False Negative) の抽出とプロット
    fn = (y == 1) & (oof["prediction"] < metrics["best_threshold"])

    shap.summary_plot(shap_values[fn], X[fn], show=False)
    plt.savefig(f"images/{exp_name}_FN_shap.png")
    plt.close()

    # 誤検出 (False Positive) の抽出
    fp = (y == 0) & (oof["prediction"] > metrics["best_threshold"])

    shap.summary_plot(shap_values[fp], X[fp], show=False)
    plt.savefig(f"images/{exp_name}_FP_shap.png")
    plt.close()

    # バープロット比較図の生成・保存
    fig, axes = plt.subplots(1, 4, figsize=(20, 15))

    plt.sca(axes[0])
    shap.summary_plot(shap_values, X, plot_type="bar", show=False)
    axes[0].set_title("All Data")

    plt.sca(axes[1])
    shap.summary_plot(
        shap_values[y == 1], X[y == 1], plot_type="bar", show=False
    )
    axes[1].set_title("y == 1")

    plt.sca(axes[2])
    shap.summary_plot(shap_values[fn], X[fn], plot_type="bar", show=False)
    axes[2].set_title("False Negative")

    plt.sca(axes[3])
    shap.summary_plot(shap_values[fp], X[fp], plot_type="bar", show=False)
    axes[3].set_title("False_Positive")

    plt.subplots_adjust(left=0.25, wspace=0.4)
    plt.savefig(f"images/{exp_name}_特徴量重要度比較.png")
    plt.close()

In [4]:
def run_oof_seed_average_shap_analysis(
    config_name: str,
    exp_dir_name: str
) -> None:

    # =========================================================
    # 1. Config / model / features / metrics / OOF をロード
    # =========================================================

    config = load_config(
        f"../configs/{config_name}.yaml"
    )

    exp_name = config["experiment"]["name"]

    model = joblib.load(
        f"../outputs/{exp_dir_name}/model.pkl"
    )

    features = joblib.load(
        f"../outputs/{exp_dir_name}/feature_columns.pkl"
    )

    with open(
        f"../outputs/{exp_dir_name}/metrics.json",
        "r",
        encoding="utf-8"
    ) as f:
        metrics = json.load(f)

    oof = pd.read_csv(
        f"../outputs/{exp_dir_name}/oof.csv"
    )

    # =========================================================
    # 2. Trainデータのロード
    # =========================================================

    train = load_train()

    target = config["data"]["target"]
    id_col = config["data"]["id"]

    cat_features = config["feature"]["categorical_features"]

    drop_cols = (
        config["feature"]["drop_columns"]
        + [target]
        + [id_col]
    )

    # 学習時と同じカテゴリ変換
    train, _ = convert_category(
        train,
        train,
        cat_features
    )

    # 学習時のXと同じ状態
    X_raw = train.drop(
        columns=drop_cols
    )

    y_raw = train[target]

    # =========================================================
    # 3. Seed / Fold設定
    # =========================================================

    n_splits = config.get(
        "train", {}
    ).get(
        "n_splits",
        5
    )

    seeds = config.get(
        "train", {}
    ).get(
        "seeds",
        [42, 52, 62, 72, 82]
    )

    # =========================================================
    # 4. モデル数の確認
    # =========================================================

    expected_models = len(seeds) * n_splits

    if len(model.models) != expected_models:
        raise ValueError(
            f"保存されているモデル数が想定と異なります。\n"
            f"Expected: {expected_models}\n"
            f"Actual  : {len(model.models)}"
        )

    print("=" * 60)
    print("OOF Seed Average SHAP Analysis")
    print("=" * 60)
    print(f"Seeds       : {seeds}")
    print(f"Folds       : {n_splits}")
    print(f"Models      : {len(model.models)}")
    print(f"Features    : {len(features)}")
    print("=" * 60)

    # =========================================================
    # 5. SHAP値の保存先
    #
    # 各企業について
    #
    #   Seed 42 → 1回
    #   Seed 52 → 1回
    #   Seed 62 → 1回
    #   Seed 72 → 1回
    #   Seed 82 → 1回
    #
    # の合計5回のSHAPを計算し、最後に平均する。
    # =========================================================

    n_samples = len(X_raw)
    n_features = len(features)

    shap_sum = np.zeros(
        (n_samples, n_features),
        dtype=float
    )

    shap_count = np.zeros(
        n_samples,
        dtype=int
    )

    # =========================================================
    # 6. SHAPプロット用のXを保存
    #
    # 文字列カテゴリがあるため、
    # np.zerosではなくDataFrameを使用する。
    #
    # Seed 42で作成されたXを代表として使用する。
    # =========================================================

    X_reference = pd.DataFrame(
        index=X_raw.index,
        columns=features,
        dtype=object
    )

    # Seed 42 のXが各行に代入されたかを記録
    X_reference_filled = np.zeros(
        n_samples,
        dtype=bool
    )

    # =========================================================
    # 7. Seed × Fold
    # =========================================================

    for seed_idx, seed in enumerate(seeds):

        print()
        print(
            "=" * 60
        )
        print(
            f"Seed {seed} "
            f"({seed_idx + 1}/{len(seeds)})"
        )
        print(
            "=" * 60
        )

        # -----------------------------------------------------
        # 学習時と完全に同じCV split
        # -----------------------------------------------------

        skf = StratifiedKFold(
            n_splits=n_splits,
            shuffle=True,
            random_state=seed
        )

        for fold, (
            train_idx,
            val_idx
        ) in enumerate(
            skf.split(X_raw, y_raw),
            start=1
        ):

            print(
                f"Seed {seed} / Fold {fold}"
            )

            # =================================================
            # 7-1. Raw data
            # =================================================

            X_tr_raw = X_raw.iloc[
                train_idx
            ].copy()

            X_val_raw = X_raw.iloc[
                val_idx
            ].copy()

            # =================================================
            # 7-2. CV特徴量
            #
            # 学習時と同じく、
            # train foldからstatsを作成し、
            # validation foldへ適用する。
            # =================================================

            X_tr, stats_dict = create_cv_features(
                X_tr_raw
            )

            X_val, _ = create_cv_features(
                X_val_raw,
                stats_dict=stats_dict
            )

            # =================================================
            # 7-3. 学習時の特徴量列・順序に合わせる
            # =================================================

            missing_features = [
                col
                for col in features
                if col not in X_val.columns
            ]

            if missing_features:
                raise ValueError(
                    f"Seed {seed}, Fold {fold} で"
                    f"以下の特徴量が存在しません:\n"
                    f"{missing_features}"
                )

            X_val = X_val[
                features
            ]

            # =================================================
            # 7-4. train / validationの特徴量確認
            # =================================================

            if list(X_tr.columns) != list(X_val.columns):

                raise ValueError(
                    f"Seed {seed}, Fold {fold} で"
                    f"train / validationの特徴量列が"
                    f"一致していません。"
                )

            # =================================================
            # 7-5. 対応するモデルを取得
            #
            # light_gbm_cv.pyでは
            #
            # models.append({
            #     "seed": seed,
            #     "fold": fold,
            #     "model": model
            # })
            #
            # となっている。
            # =================================================

            model_idx = (
                seed_idx * n_splits
                + (fold - 1)
            )

            model_info = model.models[
                model_idx
            ]

            # =================================================
            # 7-6. seed / foldの整合性確認
            # =================================================

            if model_info["seed"] != seed:

                raise ValueError(
                    f"モデルのseedが一致しません。\n"
                    f"Expected seed: {seed}\n"
                    f"Actual seed  : "
                    f"{model_info['seed']}"
                )

            if model_info["fold"] != fold:

                raise ValueError(
                    f"モデルのfoldが一致しません。\n"
                    f"Expected fold: {fold}\n"
                    f"Actual fold  : "
                    f"{model_info['fold']}"
                )

            fold_model = model_info[
                "model"
            ]

            # =================================================
            # 7-7. SHAP
            # =================================================

            explainer = shap.TreeExplainer(
                fold_model
            )

            shap_vals = explainer.shap_values(
                X_val
            )

            # =================================================
            # 7-8. SHAPの形式を統一
            # =================================================

            if isinstance(
                shap_vals,
                list
            ):
                # binary classification
                shap_vals = shap_vals[1]

            elif isinstance(
                shap_vals,
                shap.Explanation
            ):
                shap_vals = shap_vals.values

            shap_vals = np.asarray(
                shap_vals
            )

            # =================================================
            # 7-9. shape確認
            # =================================================

            expected_shape = (
                len(val_idx),
                n_features
            )

            if shap_vals.shape != expected_shape:

                raise ValueError(
                    f"\nSHAP shape mismatch\n"
                    f"Seed     : {seed}\n"
                    f"Fold     : {fold}\n"
                    f"Actual   : {shap_vals.shape}\n"
                    f"Expected : {expected_shape}\n"
                )

            # =================================================
            # 7-10. SHAPを加算
            # =================================================

            shap_sum[
                val_idx
            ] += shap_vals

            shap_count[
                val_idx
            ] += 1

            # =================================================
            # 7-11. Seed42のXを保存
            # =================================================

            if seed_idx == 0:

                X_reference.loc[
                    X_val.index,
                    features
                ] = X_val

                X_reference_filled[
                    val_idx
                ] = True
    # =========================================================
    # 8. 全企業で5 seed分のSHAPが存在するか確認
    # =========================================================

    expected_count = len(seeds)

    if not np.all(
        shap_count == expected_count
    ):

        missing_idx = np.where(
            shap_count != expected_count
        )[0]

        raise ValueError(
            "全企業について全seedのSHAPが"
            "揃っていません。\n"
            f"Expected count : {expected_count}\n"
            f"Missing rows   : {len(missing_idx)}"
        )

    # =========================================================
    # 9. SHAPのseed平均
    # =========================================================

    shap_values = (
        shap_sum
        / shap_count[:, None]
    )

    # =========================================================
    # 10. X_referenceの確認
    # =========================================================
    if not np.all(X_reference_filled):

        missing_idx = np.where(
            ~X_reference_filled
        )[0]

        raise ValueError(
            "Seed42のXが全企業について"
            "正しく保存されていません。\n"
            f"Missing rows: {len(missing_idx)}\n"
            f"Missing indices: {missing_idx[:20]}"
        )

    X = X_reference.copy()

    # 元のindexを削除して
    # y / OOFと行番号を合わせる
    X = X.reset_index(
        drop=True
    )

    y = y_raw.reset_index(
        drop=True
    )

    # =========================================================
    # 11. OOF prediction
    # =========================================================

    oof_prediction = (
        oof["prediction"]
        .to_numpy()
    )

    threshold = float(
        metrics["best_threshold"]
    )

    pred_label = (
        oof_prediction >= threshold
    )

    y_array = y.to_numpy()

    # =========================================================
    # 12. FN / FP
    # =========================================================

    fn = (
        (y_array == 1)
        & (~pred_label)
    )

    fp = (
        (y_array == 0)
        & pred_label
    )

    print()
    print("=" * 60)
    print("SHAP Summary")
    print("=" * 60)

    print(
        f"Samples        : {n_samples}"
    )

    print(
        f"Features       : {n_features}"
    )

    print(
        f"Seeds          : {len(seeds)}"
    )

    print(
        f"SHAP models    : {len(model.models)}"
    )

    print(
        f"Threshold      : {threshold:.4f}"
    )

    print(
        f"Positive       : {y_array.sum()}"
    )

    print(
        f"False Negative : {fn.sum()}"
    )

    print(
        f"False Positive : {fp.sum()}"
    )

    # =========================================================
    # 13. 全データ SHAP summary
    # =========================================================

    shap.summary_plot(
        shap_values,
        X,
        show=False
    )

    plt.savefig(
        f"images/{exp_name}_shap.png",
        bbox_inches="tight",
        dpi=300
    )

    plt.close()

    # =========================================================
    # 14. 購入企業 SHAP
    # =========================================================

    positive_mask = (
        y_array == 1
    )

    if positive_mask.sum() > 0:

        shap.summary_plot(
            shap_values[positive_mask],
            X[positive_mask],
            show=False
        )

        plt.savefig(
            f"images/{exp_name}_購入企業_shap.png",
            bbox_inches="tight",
            dpi=300
        )

        plt.close()

    # =========================================================
    # 15. False Negative SHAP
    # =========================================================

    if fn.sum() > 0:

        shap.summary_plot(
            shap_values[fn],
            X[fn],
            show=False
        )

        plt.savefig(
            f"images/{exp_name}_FN_shap.png",
            bbox_inches="tight",
            dpi=300
        )

        plt.close()

    else:

        print(
            "False Negativeが存在しないため、"
            "FN SHAPをスキップしました。"
        )

    # =========================================================
    # 16. False Positive SHAP
    # =========================================================

    if fp.sum() > 0:

        shap.summary_plot(
            shap_values[fp],
            X[fp],
            show=False
        )

        plt.savefig(
            f"images/{exp_name}_FP_shap.png",
            bbox_inches="tight",
            dpi=300
        )

        plt.close()

    else:

        print(
            "False Positiveが存在しないため、"
            "FP SHAPをスキップしました。"
        )

    # =========================================================
    # 17. SHAP bar plot
    # =========================================================

    fig, axes = plt.subplots(
        1,
        4,
        figsize=(20, 15)
    )

    # ---------------------------------------------------------
    # All Data
    # ---------------------------------------------------------

    plt.sca(axes[0])

    shap.summary_plot(
        shap_values,
        X,
        plot_type="bar",
        show=False
    )

    axes[0].set_title(
        "All Data"
    )

    # ---------------------------------------------------------
    # y == 1
    # ---------------------------------------------------------

    plt.sca(axes[1])

    if positive_mask.sum() > 0:

        shap.summary_plot(
            shap_values[positive_mask],
            X[positive_mask],
            plot_type="bar",
            show=False
        )

    axes[1].set_title(
        "y == 1"
    )

    # ---------------------------------------------------------
    # False Negative
    # ---------------------------------------------------------

    plt.sca(axes[2])

    if fn.sum() > 0:

        shap.summary_plot(
            shap_values[fn],
            X[fn],
            plot_type="bar",
            show=False
        )

    axes[2].set_title(
        "False Negative"
    )

    # ---------------------------------------------------------
    # False Positive
    # ---------------------------------------------------------

    plt.sca(axes[3])

    if fp.sum() > 0:

        shap.summary_plot(
            shap_values[fp],
            X[fp],
            plot_type="bar",
            show=False
        )

    axes[3].set_title(
        "False Positive"
    )

    plt.subplots_adjust(
        left=0.25,
        wspace=0.4
    )

    plt.savefig(
        f"images/{exp_name}_特徴量重要度比較.png",
        bbox_inches="tight",
        dpi=300
    )

    plt.close()

    # =========================================================
    # 18. SHAP値そのものを保存
    # =========================================================

    shap_df = pd.DataFrame(
        shap_values,
        columns=features
    )

    shap_df.insert(
        0,
        id_col,
        train[id_col].values
    )

    shap_df.insert(
        1,
        "target",
        y_array
    )

    shap_df.insert(
        2,
        "prediction",
        oof_prediction
    )

    shap_df.insert(
        3,
        "pred_label",
        pred_label.astype(int)
    )

    shap_df.to_csv(
        f"../outputs/{exp_dir_name}/oof_shap_values.csv",
        index=False
    )

    # =========================================================
    # 19. SHAP feature importance
    # =========================================================

    shap_importance = pd.DataFrame(
        {
            "feature": features,

            "mean_abs_shap": np.mean(
                np.abs(shap_values),
                axis=0
            ),

            "mean_shap": np.mean(
                shap_values,
                axis=0
            )
        }
    )

    shap_importance = (
        shap_importance
        .sort_values(
            "mean_abs_shap",
            ascending=False
        )
        .reset_index(drop=True)
    )

    shap_importance.to_csv(
        f"../outputs/{exp_dir_name}/shap_importance.csv",
        index=False
    )

    # =========================================================
    # 20. 完了
    # =========================================================

    print()
    print("=" * 60)
    print("SHAP analysis completed.")
    print("=" * 60)

    print(
        f"SHAP values  : "
        f"../outputs/{exp_dir_name}/oof_shap_values.csv"
    )

    print(
        f"SHAP import. : "
        f"../outputs/{exp_dir_name}/shap_importance.csv"
    )

    print(
        f"Images       : "
        f"images/{exp_name}_*.png"
    )

    print("=" * 60)

### exp001

In [ ]:
run_shap_analysis('baseline', 'exp001')

### exp002

In [17]:
run_shap_analysis('exp002', 'exp002')

/opt/anaconda3/envs/StudentCup26env/lib/python3.11/site-packages/shap/explainers/_tree.py:620: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(
/opt/anaconda3/envs/StudentCup26env/lib/python3.11/site-packages/shap/plots/_beeswarm.py:1150: UserWarning: Tight layout not applied. tight_layout cannot make Axes width small enough to accommodate all Axes decorations
  plt.tight_layout()
/opt/anaconda3/envs/StudentCup26env/lib/python3.11/site-packages/shap/plots/_beeswarm.py:1150: UserWarning: Tight layout not applied. tight_layout cannot make Axes width small enough to accommodate all Axes decorations
  plt.tight_layout()


In [19]:
shap_values, X, y, oof = calculate_shap("exp002", "exp002")

/opt/anaconda3/envs/StudentCup26env/lib/python3.11/site-packages/shap/explainers/_tree.py:620: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(


In [36]:
cols = [
    "sw支出_対_売上比",
    "sw支出_対_総資産比",
    "売上高営業利益率",
    "売上高経常利益率"
]

# 2行2列の図領域を作成 (figsizeは全体サイズに合わせて適宜調整)
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# axes を1次元配列に平坦化してループ処理を扱いやすくする
axes_flat = axes.flatten()

for i, col in enumerate(cols):
    plt.sca(axes_flat[i]) # カレントの描画先を指定
    shap.dependence_plot(
        col,
        shap_values,
        X,
        interaction_index="業界",
        ax=axes_flat[i],  # 描画対象の Subplot を渡す
        show=False        # 自動表示・破棄を防止
    )

# レイアウトを整えて保存
plt.tight_layout()
plt.savefig("images/exp002_相互shap_業界.png", dpi=300, bbox_inches="tight")
plt.close() # メモリ解放

In [39]:
cols = [
    "アンケート２",
    "アンケート４",
    "アンケート７",
    "アンケート８",
    "アンケート１０",
]

# 2行2列の図領域を作成 (figsizeは全体サイズに合わせて適宜調整)
fig, axes = plt.subplots(2, 3, figsize=(16, 12))

# axes を1次元配列に平坦化してループ処理を扱いやすくする
axes_flat = axes.flatten()

for i, col in enumerate(cols):
    plt.sca(axes_flat[i]) # カレントの描画先を指定
    shap.dependence_plot(
        "sw支出_対_売上比",
        shap_values,
        X,
        interaction_index=col,
        ax=axes_flat[i],  # 描画対象の Subplot を渡す
        show=False        # 自動表示・破棄を防止
    )

# レイアウトを整えて保存
plt.tight_layout()
plt.savefig("images/exp002_相互shap_sw支出_アンケート.png", dpi=300, bbox_inches="tight")
plt.close() # メモリ解放

### exp003

In [6]:
run_shap_analysis('exp003', 'exp003')

/opt/anaconda3/envs/StudentCup26env/lib/python3.11/site-packages/shap/explainers/_tree.py:620: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(
/opt/anaconda3/envs/StudentCup26env/lib/python3.11/site-packages/shap/plots/_beeswarm.py:1150: UserWarning: Tight layout not applied. tight_layout cannot make Axes width small enough to accommodate all Axes decorations
  plt.tight_layout()
/opt/anaconda3/envs/StudentCup26env/lib/python3.11/site-packages/shap/plots/_beeswarm.py:1150: UserWarning: Tight layout not applied. tight_layout cannot make Axes width small enough to accommodate all Axes decorations
  plt.tight_layout()
/opt/anaconda3/envs/StudentCup26env/lib/python3.11/site-packages/shap/plots/_beeswarm.py:1150: UserWarning: Tight layout not applied. tight_layout cannot make Axes width small enough to accommodate all Axes decorations
  plt.tight_layout()


### exp004

In [6]:
run_shap_analysis('exp004', 'exp004')

/opt/anaconda3/envs/StudentCup26env/lib/python3.11/site-packages/shap/explainers/_tree.py:620: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(
/opt/anaconda3/envs/StudentCup26env/lib/python3.11/site-packages/shap/plots/_beeswarm.py:1150: UserWarning: Tight layout not applied. tight_layout cannot make Axes width small enough to accommodate all Axes decorations
  plt.tight_layout()
/opt/anaconda3/envs/StudentCup26env/lib/python3.11/site-packages/shap/plots/_beeswarm.py:1150: UserWarning: Tight layout not applied. tight_layout cannot make Axes width small enough to accommodate all Axes decorations
  plt.tight_layout()
/opt/anaconda3/envs/StudentCup26env/lib/python3.11/site-packages/shap/plots/_beeswarm.py:1150: UserWarning: Tight layout not applied. tight_layout cannot make Axes width small enough to accommodate all Axes decorations
  plt.tight_layout()


### exp005

In [10]:
run_oof_shap_analysis('exp005', 'exp005')

/opt/anaconda3/envs/StudentCup26env/lib/python3.11/site-packages/shap/explainers/_tree.py:620: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(
/opt/anaconda3/envs/StudentCup26env/lib/python3.11/site-packages/shap/explainers/_tree.py:620: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(
/opt/anaconda3/envs/StudentCup26env/lib/python3.11/site-packages/shap/explainers/_tree.py:620: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(
/opt/anaconda3/envs/StudentCup26env/lib/python3.11/site-packages/shap/explainers/_tree.py:620: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(
/opt/anaconda3/envs/StudentCup26env/lib/python3.11/site-packages/shap/explainers/_tree.py:620: UserWarning: 

In [5]:
run_oof_seed_average_shap_analysis('exp006', 'exp006')

OOF Seed Average SHAP Analysis
Seeds       : [42, 52, 62, 72, 82]
Folds       : 5
Models      : 25
Features    : 79

Seed 42 (1/5)
Seed 42 / Fold 1
Seed 42 / Fold 2
Seed 42 / Fold 3


/opt/anaconda3/envs/StudentCup26env/lib/python3.11/site-packages/shap/explainers/_tree.py:620: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(
/opt/anaconda3/envs/StudentCup26env/lib/python3.11/site-packages/shap/explainers/_tree.py:620: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(
/opt/anaconda3/envs/StudentCup26env/lib/python3.11/site-packages/shap/explainers/_tree.py:620: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(
/opt/anaconda3/envs/StudentCup26env/lib/python3.11/site-packages/shap/explainers/_tree.py:620: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(


Seed 42 / Fold 4
Seed 42 / Fold 5

Seed 52 (2/5)
Seed 52 / Fold 1
Seed 52 / Fold 2


/opt/anaconda3/envs/StudentCup26env/lib/python3.11/site-packages/shap/explainers/_tree.py:620: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(
/opt/anaconda3/envs/StudentCup26env/lib/python3.11/site-packages/shap/explainers/_tree.py:620: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(
/opt/anaconda3/envs/StudentCup26env/lib/python3.11/site-packages/shap/explainers/_tree.py:620: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(
/opt/anaconda3/envs/StudentCup26env/lib/python3.11/site-packages/shap/explainers/_tree.py:620: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(


Seed 52 / Fold 3
Seed 52 / Fold 4
Seed 52 / Fold 5

Seed 62 (3/5)
Seed 62 / Fold 1


...

Seed 72 / Fold 5

Seed 82 (5/5)
Seed 82 / Fold 1
Seed 82 / Fold 2
Seed 82 / Fold 3


/opt/anaconda3/envs/StudentCup26env/lib/python3.11/site-packages/shap/explainers/_tree.py:620: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(
/opt/anaconda3/envs/StudentCup26env/lib/python3.11/site-packages/shap/explainers/_tree.py:620: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(
/opt/anaconda3/envs/StudentCup26env/lib/python3.11/site-packages/shap/explainers/_tree.py:620: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(
/opt/anaconda3/envs/StudentCup26env/lib/python3.11/site-packages/shap/explainers/_tree.py:620: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(


Seed 82 / Fold 4
Seed 82 / Fold 5

SHAP Summary
Samples        : 742
Features       : 79
Seeds          : 5
SHAP models    : 25
Threshold      : 0.2200
Positive       : 179
False Negative : 37
False Positive : 100


/opt/anaconda3/envs/StudentCup26env/lib/python3.11/site-packages/shap/explainers/_tree.py:620: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(
/opt/anaconda3/envs/StudentCup26env/lib/python3.11/site-packages/shap/plots/_beeswarm.py:1150: UserWarning: Tight layout not applied. tight_layout cannot make Axes width small enough to accommodate all Axes decorations
  plt.tight_layout()
/opt/anaconda3/envs/StudentCup26env/lib/python3.11/site-packages/shap/plots/_beeswarm.py:1150: UserWarning: Tight layout not applied. tight_layout cannot make Axes width small enough to accommodate all Axes decorations
  plt.tight_layout()
/opt/anaconda3/envs/StudentCup26env/lib/python3.11/site-packages/shap/plots/_beeswarm.py:1150: UserWarning: Tight layout not applied. tight_layout cannot make Axes width small enough to accommodate all Axes decorations
  plt.tight_layout()



SHAP analysis completed.
SHAP values  : ../outputs/exp006/oof_shap_values.csv
SHAP import. : ../outputs/exp006/shap_importance.csv
Images       : images/exp006_*.png
